# ShoeDLU Interactive Gradio Demo

This notebook provides an interactive interface for the existing ShoeDLU implementation. It uses the project's `World`, parser, dialogue-manager, and optional Stage 3 classes directly.

The interface contains:

- a button that creates a new random world
- a command field, parser selector, and optional ModelChecker switch
- a scrollable dialogue console
- an object pool showing every object that exists
- a visual world view containing the hand, toolbox, shelf slots, drying area, and floor box

The shelf visualization maps `World.shelf_slots` directly, including empty slots. Object tooltips use `World.describe_object_by_id()`. The toolbox, drying area, and floor box hide their objects visually and list the contained object IDs in their tooltips. After every command, the current mutated `World` instance is rendered again.


In [1]:
from __future__ import annotations

import html
import os
import sys
from functools import lru_cache
from pathlib import Path
from typing import Optional
from getpass import getpass

import gradio as gr


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import dialogue, parsers, stage3, world

MODEL_CHECKER_CONFIDENCE = 0.90
PARSER_NAMES = [
    "RuleBasedParser",
    "IntentClassifierParser",
    "SlotTaggerParser"
]

HF_TOKEN = os.getenv("HF_TOKEN") or getpass(
    "Enter your Hugging Face token: "
)

Enter your Hugging Face token:  ········


In [3]:
#---------------------
# Visual configuration
#---------------------

# Map domain shoe colors to browser-safe display colors
SHOE_COLORS = {
    "red": "#d84b4b",
    "green": "#4f9f62",
    "blue": "#3d78b5",
    "yellow": "#f0cf3a",
    "black": "#242424",
    "white": "#f8f8f5",
    "brown": "#8a5a37",
}

# Map symbolic shoe heights to SVG rectangle heights in pixels
SHOE_HEIGHTS = {
    "low": 30,
    "mid": 45,
    "high": 62,
}

# Styling embedded inside the generated SVG visualizations
SVG_STYLE = """
<style>
.world-card {
    border: 1px solid #c8c8c8;
    border-radius: 10px;
    background: #ffffff;
    overflow: hidden;
}
.world-card svg {
    display: block;
    width: 100%;
    height: auto;
}
.world-object {
    cursor: help;
    transition: filter 0.12s ease, opacity 0.12s ease;
}
.world-object:hover {
    filter: drop-shadow(0 0 4px rgba(28, 96, 160, 0.65));
    opacity: 0.88;
}
.location-zone {
    cursor: help;
}
.location-zone:hover rect,
.location-zone:hover circle {
    fill: #f3f7fb;
}
</style>
"""

# Styling applied to the surrounding Gradio interface
APP_CSS = """
.gradio-container {
    max-width: 1800px !important;
}
#console-output textarea {
    height: 680px !important;
    max-height: 680px !important;
    overflow-y: auto !important;
    resize: none !important;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace !important;
    white-space: pre-wrap !important;
}
#command-row {
    align-items: end;
}
#object-pool-view,
#world-view {
    min-height: 180px;
}
"""

In [4]:
#--------------
# SVG rendering
#--------------

"""Objects are rendered from their actual world class and attributes:

- shoes: colored rectangles whose height follows low, mid, or high
- cleaning utensils: light-grey circles
- impregnation utensils: light-grey triangles
- repair tools: dark-grey rectangles

The object pool always shows every existing object. The lower world view 
only draws shelf and hand objects. The storage boxes expose their 
contents through hover tooltips.
"""



def _object_tooltip(current_world: world.World, object_id: str) -> str:
    """Return the world's own object description as escaped tooltip text."""
    
    return html.escape(
        str(current_world.describe_object_by_id(object_id)), 
        quote=True
    )


    
def _render_object_svg(
    current_world: world.World,
    object_id: str,
    center_x: float,
    baseline_y: float,
    scale: float = 1.0,
    show_label: bool = False
) -> str:
    """Render one world object with the shape defined for its object class."""
    
    category = current_world.get_object_category(object_id)
    tooltip = _object_tooltip(current_world, object_id)

    # Group the shape and label into one hoverable object with a tooltip
    elements = [f'<g class="world-object"><title>{tooltip}</title>']

    match category:
        case "shoe":
            shoe = current_world.shoes[object_id]
    
            # Encode the shoe's symbolic height and color in its SVG appearance
            width = 34 * scale
            height = SHOE_HEIGHTS.get(shoe.height, 42) * scale
            x = center_x - width / 2
            y = baseline_y - height
            fill = SHOE_COLORS.get(shoe.color, "#b8b8b8")
            elements.append(
                f'<rect x="{x:.1f}" y="{y:.1f}" width="{width:.1f}" '
                f'height="{height:.1f}" rx="2" fill="{fill}" '
                'stroke="#303030" stroke-width="2" />'
            )
        case "cleaning_utensil":
            radius = 13 * scale
            elements.append(
                f'<circle cx="{center_x:.1f}" cy="{baseline_y - radius:.1f}" '
                f'r="{radius:.1f}" fill="#d4d4d4" stroke="#4a4a4a" '
                'stroke-width="2" />'
            )
        case "impregnation_utensil":
            half_width = 15 * scale
            height = 27 * scale
            points = (
                f"{center_x:.1f},{baseline_y - height:.1f} "
                f"{center_x - half_width:.1f},{baseline_y:.1f} "
                f"{center_x + half_width:.1f},{baseline_y:.1f}"
            )
            elements.append(
                f'<polygon points="{points}" fill="#d4d4d4" '
                'stroke="#4a4a4a" stroke-width="2" />'
            )
        case "repair_tool":
            width = 14 * scale
            height = 38 * scale
            elements.append(
                f'<rect x="{center_x - width / 2:.1f}" '
                f'y="{baseline_y - height:.1f}" width="{width:.1f}" '
                f'height="{height:.1f}" fill="#555b61" stroke="#252525" '
                'stroke-width="2" />'
            )

    if show_label:
        elements.append(
            f'<text x="{center_x:.1f}" y="{baseline_y + 18:.1f}" '
            'font-size="11" text-anchor="middle" fill="#333">'
            f'{html.escape(str(object_id), quote=True)}</text>'
        )
    elements.append("</g>")
    return "".join(elements)


    
def _ids_at_location(current_world: world.World, location: str) -> list[str]:
    """Return object IDs currently stored at one non-shelf location."""

    # Sort IDs to keep location tooltips stable across repeated renders
    return sorted(
        object_id
        for object_id in current_world.get_all_object_ids()
        if current_world.get_location(object_id) == location
    )


    
def _location_tooltip(
    current_world: world.World,
    label: str,
    location: str,
) -> str:
    """Return a tooltip listing the IDs hidden in a location box."""
    
    object_ids = _ids_at_location(current_world, location)

    # Display none when the location contains no objects
    contents = ", ".join(object_ids) if object_ids else "none"
    return html.escape(
        str(f"{label} ({location})\nObjects: {contents}"), 
        quote=True
    )



def render_object_pool(current_world: world.World) -> str:
    """Render miniature versions of all objects currently existing in the world."""
    
    object_ids = current_world.get_all_object_ids()

    # Arrange the objects in a responsive grid with at most eight columns
    columns = min(8, max(1, len(object_ids)))
    rows = max(1, (len(object_ids) + columns - 1) // columns)
    
    width = 900
    cell_width = width / columns
    row_height = 92
    height = 28 + rows * row_height

    elements = [
        SVG_STYLE,
        '<div class="world-card">',
        f'<svg viewBox="0 0 {width} {height}" role="img" '
        'aria-label="Object pool">',
        '<text x="18" y="21" font-size="15" font-weight="600" '
        'fill="#333">Object pool — hover over an object for details</text>'
    ]

    for index, object_id in enumerate(object_ids):
        # Map the object's linear index to its grid row and column
        row_index, column_index = divmod(index, columns)
        
        center_x = (column_index + 0.5) * cell_width
        baseline_y = 42 + row_index * row_height + 40
        elements.append(
            _render_object_svg(
                current_world=current_world,
                object_id=object_id,
                center_x=center_x,
                baseline_y=baseline_y,
                scale=0.82,
                show_label=True
            )
        )

    elements.extend(["</svg>", "</div>"])
    return "".join(elements)



def _render_location_box(
    current_world: world.World,
    label: str,
    location: str,
    x: float,
    y: float,
    width: float,
    height: float,
) -> str:
    """Render a location box whose tooltip lists its hidden object IDs."""
    
    tooltip = _location_tooltip(current_world, label, location)
    return (
        f'<g class="location-zone"><title>{tooltip}</title>'
        f'<rect x="{x:.1f}" y="{y:.1f}" width="{width:.1f}" '
        f'height="{height:.1f}" rx="4" fill="#ffffff" '
        'stroke="#666" stroke-width="2" />'
        f'<text x="{x + 12:.1f}" y="{y + 28:.1f}" font-size="16" '
        f'fill="#555">{html.escape(str(label), quote=True)}</text></g>'
    )



def render_world(current_world: world.World) -> str:
    """Render the current hand, shelf slots, and hidden storage locations."""
    
    width = 1000
    height = 650
    shelf_size = len(current_world.shelf_slots["top_shelf"])

    rack_x = 205
    rack_width = min(610, 280 + shelf_size * 33)
    rack_right = rack_x + rack_width
    rack_top = 190
    rack_bottom = 548
    shelf_y = {
        "top_shelf": 288,
        "middle_shelf": 412,
        "bottom_shelf": 536,
    }

    drying_x = rack_right + 20
    drying_width = max(120, 980 - drying_x)

    elements = [
        SVG_STYLE,
        '<div class="world-card">',
        f'<svg viewBox="0 0 {width} {height}" role="img" '
        'aria-label="Current shoe-rack world">',
        '<rect x="0" y="0" width="1000" height="650" fill="#fcfcfb" />',
        '<text x="18" y="28" font-size="17" font-weight="600" '
        'fill="#333">Current world</text>',
    ]

    # Render the hand region and the currently held object, if present
    hand_tooltip = _location_tooltip(current_world, "Hand", "hand")
    elements.append(
        f'<g class="location-zone"><title>{hand_tooltip}</title>'
        '<circle cx="105" cy="112" r="62" fill="#fff" '
        'stroke="#666" stroke-width="3" />'
        '<text x="178" y="102" font-size="16" fill="#555">Hand</text>'
        '<line x1="159" y1="104" x2="173" y2="92" '
        'stroke="#777" stroke-width="2" /></g>'
    )
    if current_world.holding is not None:
        elements.append(
            _render_object_svg(
                current_world=current_world,
                object_id=current_world.holding,
                center_x=105,
                baseline_y=135,
                scale=0.9,
            )
        )

    # Render non-shelf locations as hoverable boxes
    # -> but keep the contained objects visually hidden
    elements.append(
        _render_location_box(
            current_world,
            "Toolbox",
            "tool_area",
            x=20,
            y=320,
            width=160,
            height=220,
        )
    )
    elements.append(
        _render_location_box(
            current_world,
            "Drying area",
            "drying_area",
            x=drying_x,
            y=320,
            width=drying_width,
            height=220,
        )
    )
    elements.append(
        _render_location_box(
            current_world,
            "Floor box",
            "floor_box",
            x=20,
            y=584,
            width=960,
            height=46,
        )
    )

    # Draw a rack whose width reflects the generated shelf capacity
    rack_color = "#5b2f18"
    elements.extend(
        [
            f'<text x="{rack_x:.1f}" y="{rack_top - 18:.1f}" '
            'font-size="16" fill="#555">Rack</text>',
            f'<line x1="{rack_x:.1f}" y1="{rack_top:.1f}" '
            f'x2="{rack_right:.1f}" y2="{rack_top:.1f}" '
            f'stroke="{rack_color}" stroke-width="14" stroke-linecap="round" />',
            f'<line x1="{rack_x:.1f}" y1="{rack_top:.1f}" '
            f'x2="{rack_x:.1f}" y2="{rack_bottom:.1f}" '
            f'stroke="{rack_color}" stroke-width="14" stroke-linecap="round" />',
            f'<line x1="{rack_right:.1f}" y1="{rack_top:.1f}" '
            f'x2="{rack_right:.1f}" y2="{rack_bottom:.1f}" '
            f'stroke="{rack_color}" stroke-width="14" stroke-linecap="round" />',
        ]
    )

    # Map each shelf array directly to equally spaced visual slots
    # -> preserving object order and empty positions
    for shelf_name, line_y in shelf_y.items():
        slots = current_world.shelf_slots[shelf_name]
        elements.append(
            f'<line x1="{rack_x:.1f}" y1="{line_y:.1f}" '
            f'x2="{rack_right:.1f}" y2="{line_y:.1f}" '
            f'stroke="{rack_color}" stroke-width="12" stroke-linecap="round" />'
        )
        elements.append(
            f'<text x="{rack_x + 8:.1f}" y="{line_y - 76:.1f}" '
            'font-size="12" fill="#777">'
            f'{html.escape(shelf_name.replace("_", " "), quote=True)}</text>'
        )

        slot_width = rack_width / len(slots)
        for slot_index, object_id in enumerate(slots):
            center_x = rack_x + (slot_index + 0.5) * slot_width

            # Draw slot boundaries even when the corresponding slot is empty
            if slot_index > 0:
                divider_x = rack_x + slot_index * slot_width
                elements.append(
                    f'<line x1="{divider_x:.1f}" y1="{line_y - 86:.1f}" '
                    f'x2="{divider_x:.1f}" y2="{line_y - 9:.1f}" '
                    'stroke="#d8cec7" stroke-width="1" />'
                )

            if object_id is not None:
                elements.append(
                    _render_object_svg(
                        current_world=current_world,
                        object_id=object_id,
                        center_x=center_x,
                        baseline_y=line_y - 8,
                        scale=0.92,
                    )
                )

    elements.append(
        f'<line x1="{rack_x:.1f}" y1="{rack_bottom:.1f}" '
        f'x2="{rack_right:.1f}" y2="{rack_bottom:.1f}" '
        f'stroke="{rack_color}" stroke-width="14" stroke-linecap="round" />'
    )

    elements.extend(["</svg>", "</div>"])
    return "".join(elements)

In [5]:
#----------------------------------
# Dialogue formatting and callbacks
#----------------------------------

def _append_console(console: str, entry: str) -> str:
    """Append an entry to the dialogue history with a visual separator."""
    
    if not console.strip():
        return entry
    return f"{console.rstrip()}\n\n{'#' * 60}\n\n{entry}"



@lru_cache(maxsize=len(PARSER_NAMES))
def get_parser(parser_name: str) -> parsers.BaseParser:
    """Create each parser once so learned models are not reloaded per command."""
    
    parser_types = {
        "RuleBasedParser": parsers.RuleBasedParser,
        "IntentClassifierParser": parsers.IntentClassifierParser,
        "SlotTaggerParser": parsers.SlotTaggerParser,
    }
    try:
        return parser_types[parser_name]()
    except KeyError as error:
        raise ValueError(f"Unknown parser: {parser_name}") from error



@lru_cache(maxsize=1)
def get_model_checker() -> stage3.ModelChecker:
    """Return the lazily initialized and cached Stage 3 model checker."""
    
    if not HF_TOKEN:
        raise RuntimeError(
            "Stage 3 is unavailable because HF_TOKEN is not set. "
            "Set the environment variable, restart the kernel, and rerun the notebook."
        )
    return stage3.ModelChecker(
        hf_token=HF_TOKEN,
        verbose=False,
        confidence_threshold=MODEL_CHECKER_CONFIDENCE,
    )



def create_new_world() -> tuple[world.World, str, str, str, str]:
    """Create a random world and render its initial textual and visual state."""
    current_world = world.World.create_random()
    console = (
        "Created a new random world:\n\n"
        + "#" * 60
        + "\n"
        + current_world.describe()
    )

     # Update the session state, console, both visualizations, and command field
    return (
        current_world,
        console,
        render_object_pool(current_world),
        render_world(current_world),
        ""
    )



def execute_command(
    current_world: Optional[world.World],
    console: str,
    command: str,
    parser_name: str,
    use_model_checker: bool
) -> tuple[world.World, str, str, str, str]:
    """Execute one command and return the updated textual and visual state."""

    # Recover if the Gradio session has no initialized world
    if current_world is None:
        current_world = world.World.create_random()

    command = command.strip()
    if not command:
        return (
            current_world,
            _append_console(console, "No command entered."),
            render_object_pool(current_world),
            render_world(current_world),
            "",
        )

    try:
        parser = get_parser(parser_name)
        
        # Initialize the optional Stage 3 checker only when it is enabled
        checker = get_model_checker() if use_model_checker else None
        
        result = dialogue.interpret_and_act(
            world=current_world,
            utterance=command,
            parser=parser,
            model_checker=checker,
            intent_display=True,
        )
        entry = (
            f"> {command}\n"
            f"Parser: {parser_name}\n"
            f"Stage 3: {'enabled' if use_model_checker else 'disabled'}\n\n"
            f"{dialogue.format_dialogue_output(result)}"
        )

    # Keep the interactive session usable and display callback errors in the console
    except Exception as error:
        entry = (
            f"> {command}\n"
            f"Parser: {parser_name}\n"
            f"Stage 3: {'enabled' if use_model_checker else 'disabled'}\n\n"
            f"ERROR: {type(error).__name__}: {error}"
        )

    # Return values follow the Gradio output order (final value clears the input)
    return (
        current_world,
        _append_console(console, entry),
        render_object_pool(current_world),
        render_world(current_world),
        ""
    )

In [6]:
#---------------------------
# Build the Gradio interface
#---------------------------

def build_demo() -> gr.Blocks:
    """Build and wire the two-column ShoeDLU Gradio interface."""
    
    with gr.Blocks(
        title="ShoeDLU Interactive World",
        fill_width=True,
    ) as demo:
        # Store the mutable world separately from the visible UI components
        world_state = gr.State(value=None)

        gr.Markdown(
            "# ShoeDLU Interactive World \n"
            "Create a random world, choose a parser, and execute grounded "
            "natural-language commands. Hover over objects and location boxes "
            "in the visualization for details."
        )

        # Place controls and dialogue on the left and visualizations on the right
        with gr.Row(equal_height=True):
            with gr.Column(scale=1, min_width=560):
                create_world_button = gr.Button(
                    "Create new random world",
                    variant="primary",
                    size="lg",
                )

                with gr.Row(elem_id="command-row"):
                    command_input = gr.Textbox(
                        placeholder="Enter a command",
                        label="Command",
                        lines=1,
                        submit_btn=False,
                        scale=5,
                        min_width=260,
                        autofocus=True,
                    )
                    parser_picker = gr.Dropdown(
                        choices=PARSER_NAMES,
                        value="RuleBasedParser",
                        label="Parser",
                        scale=2,
                        min_width=190,
                    )
                    model_checker_toggle = gr.Checkbox(
                        value=False,
                        label=(
                            "Use LLM"
                            if HF_TOKEN
                            else "ModelChecker unavailable: set HF_TOKEN"
                        ),
                        interactive=bool(HF_TOKEN),
                        scale=2,
                        min_width=180,
                    )
                    execute_button = gr.Button(
                        "Execute",
                        variant="secondary",
                        scale=1,
                        min_width=100,
                    )

                console_output = gr.Textbox(
                    label="Dialogue console",
                    lines=30,
                    max_lines=30,
                    interactive=False,
                    autoscroll=True,
                    elem_id="console-output",
                )

            with gr.Column(scale=1, min_width=620):
                object_pool_view = gr.HTML(
                    value="",
                    elem_id="object-pool-view",
                    min_height=180,
                    container=False,
                )
                world_view = gr.HTML(
                    value="",
                    elem_id="world-view",
                    min_height=430,
                    container=False,
                )

        shared_outputs = [
            world_state,
            console_output,
            object_pool_view,
            world_view,
            command_input,
        ]

        # Initialize a world on page load and allow the user to replace it
        create_world_button.click(
            fn=create_new_world,
            outputs=shared_outputs,
            queue=False,
        )
        demo.load(
            fn=create_new_world,
            outputs=shared_outputs,
            queue=False,
        )

        # Reuse the same callback inputs for button clicks and Enter submissions
        command_inputs = [
            world_state,
            console_output,
            command_input,
            parser_picker,
            model_checker_toggle,
        ]
        execute_button.click(
            fn=execute_command,
            inputs=command_inputs,
            outputs=shared_outputs,
            concurrency_limit=1,
        )
        command_input.submit(
            fn=execute_command,
            inputs=command_inputs,
            outputs=shared_outputs,
            concurrency_limit=1,
        )

    return demo


demo = build_demo()

In [7]:
# Queue callbacks one at a time to protect the shared world state
# -> open the application in the default browser instead of embedding it
demo.queue(default_concurrency_limit=1).launch(
    inline=False,
    inbrowser=True,
    show_error=True,
    theme=gr.themes.Soft(),
    css=APP_CSS,
)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created TensorFlow Lite XNNPACK delegate for CPU.
[22557:22587:0730/222328.955744:ERROR:google_apis/gcm/engine/registration_request.cc:291] Registration response error message: DEPRECATED_ENDPOINT
[22557:22587:0730/222356.128207:ERROR:google_apis/gcm/engine/registration_request.cc:291] Registration response error message: DEPRECATED_ENDPOINT
[22557:22587:0730/222438.200970:ERROR:google_apis/gcm/engine/registration_request.cc:291] Registration response error message: DEPRECATED_ENDPOINT
